In [1]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

In [2]:
import pandas as pd
from classes.trading.actionPredictionTrading import ActionPredictionTrading

# Load the dataset
csv_path = "../../datasets/b3_dados/processed/acoes_concat.csv"
df = pd.read_csv(csv_path, parse_dates=['Date'])

test_start_date = '2019-01-01'

# Filtra o conjunto de teste
test_data = df[df['Date'] >= test_start_date]

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [3]:
# 

def print_detailed_performance(strategy_name: str, result_dict: dict, initial_capital: float = 100000.0):
    """
    Pega o dicionário da simulação e imprime um resumo detalhado e formatado.
    """
    # Extrai e calcula as métricas pedidas
    total_trades = result_dict.get('total_trades', 0)
    if total_trades == 0:
        print(f"\n--- Análise de Performance para '{strategy_name}' ---")
        print("    - Nenhuma operação foi realizada.")
        print("-" * 50)
        return

    hit_rate = result_dict['hit_rate']
    final_capital = result_dict['final_capital']
    max_drawdown = result_dict['max_drawdown']

    dias_de_lucro = int(total_trades * hit_rate)
    dias_de_prejuizo = total_trades - dias_de_lucro
    lucro_total_rs = final_capital - initial_capital

    print(f"\n--- Análise de Performance para '{strategy_name}' ---")
    
    print("\n  Resultado Financeiro:")
    print(f"    - Lucro/Prejuízo Total: R$ {lucro_total_rs:,.2f}")
    print(f"    - Capital Final:        R$ {final_capital:,.2f}")

    print("\n  Consistência da Estratégia:")
    print(f"    - Dias de Lucro:        {dias_de_lucro}")
    print(f"    - Dias de Prejuízo:     {dias_de_prejuizo}")
    print(f"    - Total de Trades:      {total_trades}")
    print(f"    - Taxa de Acerto (Hit Rate): {hit_rate:.2%}")

    print("\n  Análise de Risco:")
    print(f"    - Máximo Drawdown:      {max_drawdown:.2%}")
    print("-" * 50)


In [4]:


# List of stocks and paths to models and scalers
stocks_models_scalers = {


    "VALE3": {
        "model_path": "../../saved_models/linear_regression/VALE3_model_v1.0.pkl",
        "scaler_x_path": "../../saved_models/linear_regression/VALE3_scaler_X_v1.0.pkl",
        "scaler_y_path": "../../saved_models/linear_regression/VALE3_scaler_y_v1.0.pkl"
    },

    # "ITUB4": {
    #     "model_path": "../../saved_models/linear_regression/ITUB4_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/ITUB4_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/ITUB4_scaler_y_v1.0.pkl"
    # },

    # "BBAS3": {
    #     "model_path": "../../saved_models/linear_regression/BBAS3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/BBAS3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/BBAS3_scaler_y_v1.0.pkl"
    # },

    # "CYRE3": {
    #     "model_path": "../../saved_models/linear_regression/CYRE3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/CYRE3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/CYRE3_scaler_y_v1.0.pkl"
    # },

    # "TEND3": {
    #     "model_path": "../../saved_models/linear_regression/TEND3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/TEND3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/TEND3_scaler_y_v1.0.pkl"
    # },

    # "DIRR3": {
    #     "model_path": "../../saved_models/linear_regression/DIRR3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/DIRR3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/DIRR3_scaler_y_v1.0.pkl"
    # },

    "ELET3": {
        "model_path": "../../saved_models/linear_regression/ELET3_model_v1.0.pkl",
        "scaler_x_path": "../../saved_models/linear_regression/ELET3_scaler_X_v1.0.pkl",
        "scaler_y_path": "../../saved_models/linear_regression/ELET3_scaler_y_v1.0.pkl"
     },

    # "EQTL3": {
    #     "model_path": "../../saved_models/linear_regression/EQTL3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/EQTL3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/EQTL3_scaler_y_v1.0.pkl"
    # },

    # "CMIG4": {
    #     "model_path": "../../saved_models/linear_regression/CMIG4_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/CMIG4_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/CMIG4_scaler_y_v1.0.pkl"
    # },

    # "PETR3": {
    #     "model_path": "../../saved_models/linear_regression/PETR3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/PETR3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/PETR3_scaler_y_v1.0.pkl"
    # },

    # "BRAP3": {
    #     "model_path": "../../saved_models/linear_regression/BRAP3_model_v1.0.pkl",
    #     "scaler_x_path": "../../saved_models/linear_regression/BRAP3_scaler_X_v1.0.pkl",
    #     "scaler_y_path": "../../saved_models/linear_regression/BRAP3_scaler_y_v1.0.pkl"
    # }




}


In [5]:
# Iterate over each stock
# Definir períodos conforme Moura (2023)
periods = {
    "pre_pandemia": ("2019-01-01", "2019-12-31"),
    "durante_pandemia": ("2020-01-01", "2021-08-31"),
    "pos_pandemia": ("2021-09-01", "2022-09-30")  # ou df['Date'].max() se quiser ir até o fim dos dados
}

# Loop por período
results = {}

for period_name, (start_date, end_date) in periods.items():
    print(f"\n===== Analisando período: {period_name.replace('_',' ').title()} =====")
    period_data = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

    for stock, paths in stocks_models_scalers.items():
        print(f"\nAnalyzing stock: {stock}")
        analysis = ActionPredictionTrading(
            period_data,
            stock,
            window=3,
            model_path=paths["model_path"],
            scaler_x_path=paths.get("scaler_x_path"),
            scaler_y_path=paths.get("scaler_y_path"),
        )
        analysis.load_model()
        analysis.generate_predictions()

        # --- Cálculo de métricas de regressão para as previsões ---
        y_true = analysis.df['actual'].values
        y_pred = analysis.df['predicted'].values
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        regression_metrics = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        }
        print(f"Regression metrics: R2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, MSE={mse:.4f}")

        result_no_stop = analysis.simulate_trading(
            stop_loss=False, stop_type='percent', stop_value=0.02
        )
        result_with_stop = analysis.simulate_trading(
            stop_loss=True, stop_type='percent', stop_value=0.03
        )
        result_bh = analysis.simulate_buy_and_hold()

        print_detailed_performance("Estratégia SEM Stop Loss", result_no_stop)
        print_detailed_performance("Estratégia COM Stop Loss", result_with_stop)

        # inclui métricas de regressão no resultado
        results[(stock, period_name)] = {
            'regression_metrics': regression_metrics,
            'no_stop_loss': result_no_stop,
            'with_stop_loss': result_with_stop,
            'buy_and_hold': result_bh
        }

# Exibir os resultados
# for (stock, period_name), result in results.items():
#     title = period_name.replace('_', ' ').title()
#     print(f"\nResults for {stock} - {title}:")
#     # Métricas de regressão
#     rm = result['regression_metrics']
#     print(f"Regression Metrics: R2={rm['R2']:.4f}, RMSE={rm['RMSE']:.4f}, MAE={rm['MAE']:.4f}, MSE={rm['MSE']:.4f}")
#     # Estratégias de trading
#     print(f"No Stop Loss: {result['no_stop_loss']}")
#     print(f"With Stop Loss: {result['with_stop_loss']}")
#     print(f"Buy and Hold: {result['buy_and_hold']}")





===== Analisando período: Pre Pandemia =====

Analyzing stock: VALE3
Scaler X loaded and validated for window=3
Scaler Y loaded and validated.
Model loaded and validated from ../../saved_models/linear_regression/VALE3_model_v1.0.pkl
Regression metrics: R2=0.9988, RMSE=0.0638, MAE=0.0474, MSE=0.0041

--- Análise de Performance para 'Estratégia SEM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ -378.99
    - Capital Final:        R$ 99,621.01

  Consistência da Estratégia:
    - Dias de Lucro:        121
    - Dias de Prejuízo:     123
    - Total de Trades:      244
    - Taxa de Acerto (Hit Rate): 49.59%

  Análise de Risco:
    - Máximo Drawdown:      1.44%
--------------------------------------------------

--- Análise de Performance para 'Estratégia COM Stop Loss' ---

  Resultado Financeiro:
    - Lucro/Prejuízo Total: R$ 98.28
    - Capital Final:        R$ 100,098.28

  Consistência da Estratégia:
    - Dias de Lucro:        121
    - Dias de Prejuízo:   

In [11]:
flat_results = []
for (stock, period_name), metrics in results.items():
    row = {
        'stock': stock,
        'period': period_name,
        # Regressão
        'MSE': metrics['regression_metrics']['MSE'],
        'RMSE': metrics['regression_metrics']['RMSE'],
        'MAE': metrics['regression_metrics']['MAE'],
        'R2': metrics['regression_metrics']['R2'],
        # Sem stop loss
        'retorno_no_stop': metrics['no_stop_loss']['total_return'],
        'acerto_no_stop': metrics['no_stop_loss']['hit_rate'],
        'sharpe_no_stop': metrics['no_stop_loss']['sharpe_ratio'],
        'drawdown_no_stop': metrics['no_stop_loss']['max_drawdown'],
        'capital_no_stop': metrics['no_stop_loss']['final_capital'],
        # Com stop loss
        'retorno_stop': metrics['with_stop_loss']['total_return'],
        'acerto_stop': metrics['with_stop_loss']['hit_rate'],
        'sharpe_stop': metrics['with_stop_loss']['sharpe_ratio'],
        'drawdown_stop': metrics['with_stop_loss']['max_drawdown'],
        'capital_stop': metrics['with_stop_loss']['final_capital'],
        # Buy & Hold
        'retorno_bh': metrics['buy_and_hold']['total_return'],
        'capital_bh': metrics['buy_and_hold']['final_capital'],
        'dias_bh': metrics['buy_and_hold']['days_held']
    }
    flat_results.append(row)

# Monta DataFrame
df_results = pd.DataFrame(flat_results)

# Salvar resultados
csv_path = "../../datasets/trading/trading_resultsv2.csv"
if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
    try:
        df_exist = pd.read_csv(csv_path)
        df_comb = pd.concat([df_exist, df_results], ignore_index=True)
        df_comb.drop_duplicates(['stock', 'period'], keep='last', inplace=True)
        df_comb.to_csv(csv_path, index=False)
    except pd.errors.EmptyDataError:
        # File exists but is empty, treat as if it doesn't exist
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        df_results.to_csv(csv_path, index=False)
else:
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df_results.to_csv(csv_path, index=False)

print(f"Resultados salvos em: {csv_path}")

# Visualização rápida
df_results.head()

Resultados salvos em: ../../datasets/trading/trading_resultsv2.csv


,stock,period,MSE,RMSE,MAE,R2,retorno_no_stop,acerto_no_stop,sharpe_no_stop,drawdown_no_stop,capital_no_stop,retorno_stop,acerto_stop,sharpe_stop,drawdown_stop,capital_stop,retorno_bh,capital_bh,dias_bh
0,VALE3,pre_pandemia,0.004076,0.063842,0.047409,0.998834,-0.003790,0.495902,-0.019765,0.014404,99621.009064,0.000983,0.495902,0.005316,0.010696,100098.278265,0.002262,100226.158714,248
1,VALE3,durante_pandemia,0.018748,0.136923,0.108925,0.999948,-0.055949,0.447433,-0.109275,0.076287,94405.082321,-0.015282,0.447433,-0.034498,0.045722,98471.795389,0.035395,103539.482117,413
2,VALE3,pos_pandemia,0.027290,0.165196,0.142957,0.999487,0.022379,0.533835,0.054773,0.023760,102237.891769,0.040988,0.533835,0.109245,0.017692,104098.799450,-0.011628,98837.223434,270


In [15]:
df_results[["MSE", "RMSE", "MAE", "R2"]]

,MSE,RMSE,MAE,R2
0,0.004076,0.063842,0.047409,0.998834
1,0.018748,0.136923,0.108925,0.999948
2,0.027290,0.165196,0.142957,0.999487
